XML → Clean → RDF Triples → Graph → PageRank
        ↓
   Node Embeddings (FAISS)
        ↓
Retriever (Vector + Graph + Multi-hop)
        ↓
Context → LLM Answer
        ↓
Graph Visualization (NetworkX)

In [ ]:
%pip install lxml bs4 langchain langchain-openai faiss-cpu networkx matplotlib
dbutils.library.restartPython()

In [ ]:
import re
import networkx as nx
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from pyspark.sql import Row
from pyspark.sql.functions import monotonically_increasing_id

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.schema import Document

In [ ]:
xml_paths = [
    "/Volumes/...3829c0e9.../WebHome.xml",
    "/Volumes/...76f049ea.../WebHome.xml",
    "/Volumes/...07730902.../WebHome.xml",
    "/Volumes/...f2280f6d.../WebHome.xml",
    "/Volumes/...96bf76a9.../WebHome.xml"
]

rows = []

for path in xml_paths:
    with open(path, "rb") as f:
        soup = BeautifulSoup(f.read(), "xml")

    content = soup.find("content").text if soup.find("content") else ""

    xwikidoc = soup.find("xwikidoc")
    reference = xwikidoc.get("reference") if xwikidoc else None

    rows.append(Row(
        content=content,
        reference=reference,
        source=path
    ))

df = spark.createDataFrame(rows)
df = df.withColumn("doc_id", monotonically_increasing_id())

display(df)

In [ ]:
def clean_text(text):
    text = re.sub(r"<!\[CDATA\[|\]\]>", "", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\[\[.*?\]\]|\{\{.*?\}\}", "", text)
    return re.sub(r"\s+", " ", text).strip()

df_clean = df.rdd.map(lambda r: Row(
    doc_id=str(r["doc_id"]),
    content=clean_text(r["content"]),
    reference=r["reference"]
)).toDF()

display(df_clean)

In [ ]:
CELL 5 — RDF TRIPLES
triples = []

rows = df_clean.collect()

for r in rows:
    if r["reference"]:
        triples.append((r["doc_id"], "links_to", r["reference"]))

print("Triples:", triples[:5])

In [ ]:
G = nx.DiGraph()

for r in rows:
    G.add_node(r["doc_id"], content=r["content"])

for s, p, o in triples:
    if o in G.nodes():
        G.add_edge(s, o, relation=p)

print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))

In [ ]:
pagerank = nx.pagerank(G)

for node in G.nodes():
    G.nodes[node]["pagerank"] = pagerank.get(node, 0)

print("PageRank added")

In [ ]:
embedding_model = OpenAIEmbeddings()

documents = []

for r in rows:
    documents.append(
        Document(
            page_content=r["content"],
            metadata={"doc_id": r["doc_id"]}
        )
    )

vector_store = FAISS.from_documents(documents, embedding_model)

print("Vector store ready")

In [ ]:
def graph_retriever(graph, vector_store, query, max_hops=2, top_k=5):

    docs = vector_store.similarity_search(query, k=top_k)

    start_nodes = [d.metadata["doc_id"] for d in docs]
    print("Start Nodes:", start_nodes)

    visited = set()
    scores = {}
    path = []

    queue = [(node, 0) for node in start_nodes]

    for node in start_nodes:
        scores[node] = graph.nodes[node]["pagerank"]

    while queue:
        current, hop = queue.pop(0)

        if hop >= max_hops or current in visited:
            continue

        visited.add(current)
        path.append(current)

        for neighbor in graph.neighbors(current):

            pr = graph.nodes[neighbor]["pagerank"]
            score = pr / (hop + 1)

            if neighbor not in scores or score > scores[neighbor]:
                scores[neighbor] = score

            queue.append((neighbor, hop + 1))

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_nodes = [n for n, _ in ranked[:top_k]]

    print("Top Nodes:", top_nodes)

    return top_nodes, path

In [ ]:
def visualize_graph(graph, path=None):

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(graph, seed=42)

    nx.draw(graph, pos, with_labels=True,
            node_color="lightblue", node_size=2000)

    # RDF labels
    edge_labels = nx.get_edge_attributes(graph, 'relation')
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)

    if path and len(path) > 1:
        edges = list(zip(path, path[1:]))

        nx.draw_networkx_edges(
            graph, pos,
            edgelist=edges,
            edge_color="red",
            width=3
        )

        nx.draw_networkx_nodes(
            graph, pos,
            nodelist=path,
            node_color="orange"
        )

    plt.title("RDF Graph (Red = Traversal Path)")
    plt.show()

In [ ]:
class GraphRAG:

    def __init__(self, graph, vector_store, llm):
        self.graph = graph
        self.vector_store = vector_store
        self.llm = llm

    def answer(self, query):

        nodes, path = graph_retriever(self.graph, self.vector_store, query)

        visualize_graph(self.graph, path)

        context = ""
        for n in nodes:
            context += "\n" + self.graph.nodes[n]["content"]

        prompt = f"""
        Answer using context:

        {context}

        Question: {query}
        """

        return self.llm.invoke(prompt)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag = GraphRAG(G, vector_store, llm)

response = rag.answer("Explain the system")

print(response)